### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json

DATA_ROOT = next((p for p in (Path("../../data"), Path("data"), Path("downloads/python/data")) if p.exists()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "greek_sites.csv": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp05/greek_sites.csv"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix.lower() == ".gpkg":
            return path.read_bytes()[:16] == b"SQLite format 3\x00"
        if path.suffix.lower() == ".geojson":
            json.loads(path.read_text(encoding="utf-8"))
        return True
    except Exception:
        return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp05" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}")
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists():
            target.unlink()
        print(f"[WARN] Δεν ήταν δυνατή η λήψη {url}: {exc}")


# GP05 — CRS Without Fear

Συμπλήρωσε τα κενά και κράτησε σημειώσεις για το τι αλλάζει όταν μετασχηματίζουμε CRS. Οι αριθμοί δεν αρκούν: πρέπει να ξέρεις αν είναι degrees ή metres και αν το CRS δηλώθηκε ή μετασχηματίστηκε.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path


In [ ]:
data_path = DATA_ROOT / "gp05/greek_sites.csv"
df = pd.read_csv(data_path)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['longitude'], df['latitude']), crs='EPSG:____')
gdf.head()


## Έλεγχος αρχικού CRS
Εκτύπωσε CRS, πλήθος γραμμών, τύπους γεωμετρίας και bounds.

In [ ]:
print('CRS:', gdf.crs)
print('rows:', len(gdf))
print('geometry types:', gdf.geom_type.value_counts().to_dict())
print('bounds:', gdf.total_bounds)


In [ ]:
# Μετασχημάτισε σε Greek Grid για εργασία σε μέτρα
projected = gdf.to_crs('EPSG:____')
projected[['name', 'geometry']]


## QA και σύγκριση bounds
Έλεγξε ότι δεν άλλαξαν οι γραμμές ή η γεωμετρία και εξήγησε γιατί άλλαξαν οι αριθμητικές τιμές των bounds.

Controlled wrong-CRS example: δημιούργησε προσωρινά `wrong = gdf.set_crs('EPSG:2100', allow_override=True)` και σύγκρινε bounds με το σωστό `projected`. Τι αποδεικνύει ότι οι αριθμοί έμειναν ίδιοι αλλά η ερμηνεία έγινε λάθος;

In [ ]:
assert len(projected) == len(gdf)
assert projected.crs is not None
assert projected.geom_type.eq('Point').all()
print('initial bounds:', gdf.total_bounds)
print('projected bounds:', projected.total_bounds)


In [ ]:
# Mini challenge: υπολόγισε την απόσταση Αθήνας–Θεσσαλονίκης σε μέτρα
# distance_m = ...
# print(distance_m / 1000)
